# Playground — tweak the parameters and run the searches yourself

Every search is a **grid of parameter combinations** evaluated by one primitive:
`experiments.grid_full(fn, param_grid, R, phi_min, phi_max, eps, seed)` → `(success_rate, mean_budget,
params)` for every combination. Both the exploration- and exploitation-shot axes are `np.geomspace`;
you just choose **how many values** to generate. Edit any **# ---- EDIT ----** cell and re-run.

The authoritative paper tables are in `reproduce.ipynb` / `results/RESULTS.md`; this is a sandbox.

In [1]:
import numpy as np, pandas as pd
from qmetrology import experiments as E, conv95, algorithms as alg
pd.set_option('display.width', 220); pd.set_option('display.max_columns', None)

def run_grid(fn, param_grid, R=2000, phi_min=0.01, phi_max=0.1, eps=1e-3, seed=42):
    res = E.grid_full(fn, param_grid, R, phi_min, phi_max, eps, seed)
    df = pd.DataFrame([{**p, 'success_%': round(s*100, 1), 'mean_budget': round(b)} for s, b, p in res])
    print(f'{int(np.prod([len(v) for v in param_grid.values()]))} configs x R={R} trials')
    return df

## A. Fixed-budget (=10,000) grid search — the old workflow
Pick how many exploration-shot values to generate, run, sort by `success_%`, read off the best.

In [2]:
from qmetrology.algorithms import find_phi_fixed_budget_linear_search
# ---- EDIT ----
n_exploration = 25                                  # how many m_exploration values (geomspace)
param_grid = {
    'm_exploration': np.unique(np.geomspace(5, 1000, n_exploration).astype(int)),
    'budget':        [10_000],
    'lookback_window': [1, 2, 5],
    'safeguard':     [1, 2],
    'inc':           [1, 2],
}
# --------------
df = run_grid(find_phi_fixed_budget_linear_search, param_grid, R=2000, phi_max=0.1)
df.sort_values('success_%', ascending=False).head(12)

300 configs x R=2000 trials


,m_exploration,budget,lookback_window,safeguard,inc,success_%,mean_budget
13,6,10000,1,1,2,58.5,9990
15,6,10000,1,2,2,58.2,9989
25,7,10000,1,1,2,57.4,9990
1,5,10000,1,1,2,57.3,9996
37,9,10000,1,1,2,57.3,9994
49,12,10000,1,1,2,57.0,9995
264,643,10000,1,1,1,57.0,19933
266,643,10000,1,2,1,57.0,19933
27,7,10000,1,2,2,57.0,9992
19,6,10000,2,2,2,56.7,9992


Validate the winner on a fresh seed at high R (removes the winner's-curse bias):

In [3]:
best = df.sort_values('success_%', ascending=False).iloc[0].to_dict()
params = {k: int(best[k]) for k in param_grid}
val = E.success_rate(find_phi_fixed_budget_linear_search, params, R=50_000, phi_min=0.01, phi_max=0.1, eps=1e-3, seed=2024)
print('tuning  :', best['success_%'], '%   ->   validated (R=50k):', round(val*100, 1), '%')
print('config  :', params)

tuning  : 58.5 %   ->   validated (R=50k): 56.8 %
config  : {'m_exploration': 6, 'budget': 10000, 'lookback_window': 1, 'safeguard': 1, 'inc': 2}


## B. The >95%-budget search — resolution is two knobs, grids are exposed
`conv95.N_EXPLORE` / `conv95.N_EXPLOIT` set how many geomspace values each axis gets; the shape
params are `conv95.SHAPE_LINEAR` etc. Change any of them and re-run `budget95`.

In [4]:
print('N_EXPLORE =', conv95.N_EXPLORE, '  N_EXPLOIT =', conv95.N_EXPLOIT, '  EXPLORE_RANGE =', conv95.EXPLORE_RANGE)
print('SHAPE_LINEAR :', conv95.SHAPE_LINEAR)
for a in ['linear', 'binary', 'reverse_eng']:
    print(f'  grid size ({a}) = {conv95.grid_size(a):,} configs/column')

N_EXPLORE = 12   N_EXPLOIT = 24   EXPLORE_RANGE = (5, 5000)
SHAPE_LINEAR : {'lookback_window': [1, 2, 3, 5], 'safeguard': [0, 1, 2], 'inc': [1, 2]}
  grid size (linear) = 6,912 configs/column
  grid size (binary) = 7,776 configs/column
  grid size (reverse_eng) = 1,152 configs/column


In [8]:
# ---- EDIT: crank the resolution up or down, then re-run ----
conv95.N_EXPLORE = 8
conv95.N_EXPLOIT = 20
algo = 'binary'
# -----------------------------------------------------------
r = conv95.budget95(algo, eps=1e-3, phi_min=0.01, phi_max=0.1)
print(f'honest >95% budget "{algo}":', f"{r['budget']:,.0f}", '(validated', round(r['success']*100,1), '%)  ->', r['params'])

honest >95% budget "binary": 137,115 (validated 95.0 %)  -> {'m_exploration': np.int64(96), 'm_exploitation': 3902, 'conf': 0.5, 'safeguard': 2}


## C. Fully manual >95% search (see every step)
Grid over shape **and** the exploitation-budget axis, keep configs reaching 95%, take the cheapest.
(`budget95` also re-validates the winner on a second seed — do that before trusting a number.)

In [6]:
from qmetrology.algorithms import find_phi_linear_search
# ---- EDIT ----
n_exploit = 20
grid = {
    'm_exploration':   np.unique(np.geomspace(10, 500, 6).astype(int)),
    'm_exploitation':  np.unique(np.geomspace(500, 20_000, n_exploit).astype(int)),
    'lookback_window': [1, 2],
    'safeguard':       [1, 2],
    'inc':             [1],
}
# --------------
df = run_grid(find_phi_linear_search, grid, R=1500, phi_max=0.1)
df[df['success_%'] >= 95].sort_values('mean_budget').head(10)

480 configs x R=1500 trials


,m_exploration,m_exploitation,lookback_window,safeguard,inc,success_%,mean_budget
127,21,4231,2,2,1,95.1,73331
364,228,4231,1,1,1,95.5,74093
289,104,5138,1,2,1,95.1,76910
46,10,4231,2,1,1,95.1,77431
126,21,4231,2,1,1,95.9,77562
128,21,5138,1,1,1,96.2,77892
48,10,5138,1,1,1,95.7,77968
208,47,5138,1,1,1,95.9,79086
206,47,4231,2,1,1,96.0,80642
288,104,5138,1,1,1,96.6,82048


_Swap in any `find_phi_*` from `qmetrology.algorithms`; change `eps` (1e-3/1e-4), `phi_max`
(e.g. `np.pi/2`), `R`, `seed`. See `results/RESULTS.md` for the reproduced paper tables._